# Role 3: Classical & Naive Models Lead (`KO`)
- **Benchmark:** Naive (zero-change / random walk)
- **Classical Model:** Walk-forward `ARIMA(2, 0, 0)` (order tuned strictly on pre-test `INITIAL_TRAIN` data via AIC)
- **Evaluation:** RMSE, MAE, MAPE, Directional Accuracy, Out-of-Sample $R^2$, and Diebold-Mariano significance test

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import warnings
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX

from tslab.config import TARGET_TICKER, INITIAL_TRAIN, TEST_SIZE, OUTPUT_DIR
from tslab.data import load_market_data
from tslab.features import make_dataset
from tslab.validation import walk_forward_splits, describe_splits
from tslab.metrics import compare_models, diebold_mariano, dm_verdict, directional_accuracy_test

warnings.filterwarnings("ignore")

frame = load_market_data()
dataset = make_dataset(frame, horizon=1).dropna(subset=["target_return"])
splits = list(walk_forward_splits(len(dataset), initial_train=INITIAL_TRAIN, test_size=TEST_SIZE))
print(f"Ticker: {TARGET_TICKER} | Scored rows: {len(dataset)} | Folds: {len(splits)}")
display(describe_splits(splits, dataset.index).head())

Loaded 2981 trading days of KO, 2015-01-02 to 2026-11-09, with 7 predictor series.
Ticker: KO | Scored rows: 2781 | Folds: 29


,train_rows,train_from,train_to,test_rows,test_from,test_to
fold,,,,,,
1,1000,2015-10-16,2019-10-07,63,2019-10-08,2020-01-07
2,1063,2015-10-16,2020-01-07,63,2020-01-08,2020-04-07
3,1126,2015-10-16,2020-04-07,63,2020-04-08,2020-07-08
4,1189,2015-10-16,2020-07-08,63,2020-07-09,2020-10-06
5,1252,2015-10-16,2020-10-06,63,2020-10-07,2021-01-06


In [2]:
# 1. Pre-test AIC selection (first INITIAL_TRAIN rows only)
pre_test_returns = dataset["target_return"].iloc[:INITIAL_TRAIN].values
best_aic, best_order = np.inf, (1, 0, 1)
for p in range(3):
    for q in range(3):
        if p == 0 and q == 0:
            continue
        try:
            res = SARIMAX(pre_test_returns, order=(p, 0, q), trend="c", enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
            if res.aic < best_aic:
                best_aic, best_order = res.aic, (p, 0, q)
        except Exception:
            continue

print(f"Selected ARIMA order (pre-test AIC={best_aic:.2f}): {best_order}")

# 2. Walk-forward 1-step-ahead forecasting
model_name = f"ARIMA{best_order}"
arima_preds = pd.Series(np.nan, index=dataset.index, name=model_name)
for train_idx, test_idx in splits:
    y_train = dataset["target_return"].iloc[train_idx].values
    y_test = dataset["target_return"].iloc[test_idx].values
    fitted = SARIMAX(y_train, order=best_order, trend="c", enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    fold_preds, res = [], fitted
    for actual_next in y_test:
        fold_preds.append(float(res.forecast(steps=1)[0]))
        res = res.append([actual_next], refit=False)
    arima_preds.iloc[test_idx] = fold_preds

Selected ARIMA order (pre-test AIC=-6556.98): (2, 0, 0)


In [3]:
# 3. Evaluation metrics & Diebold-Mariano significance test
metrics_df = compare_models(dataset, predictions={model_name: arima_preds}, naive_name="Naive (no change)")
display(metrics_df)

scored = dataset.loc[arima_preds.notna()]
hit_rate, binom_pval = directional_accuracy_test(scored["target_return"].values, arima_preds.loc[scored.index].values)
print(f"{model_name} Directional Hit Rate: {hit_rate:.4f} (Binomial p-value vs 0.50: {binom_pval:.4f})")

actual_close = scored["target_close"].values
naive_close = scored["close"].values
arima_close = scored["close"].values * np.exp(arima_preds.loc[scored.index].values)

dm_stat_price, dm_pval_price = diebold_mariano(actual_close, arima_close, naive_close, horizon=1)
dm_stat_ret, dm_pval_ret = diebold_mariano(scored["target_return"].values, arima_preds.loc[scored.index].values, np.zeros(len(scored)), horizon=1)
print(f"Price Errors  -> DM Stat: {dm_stat_price:.4f}, p-value: {dm_pval_price:.4f} | {dm_verdict(dm_stat_price, dm_pval_price)}")
print(f"Return Errors -> DM Stat: {dm_stat_ret:.4f}, p-value: {dm_pval_ret:.4f} | {dm_verdict(dm_stat_ret, dm_pval_ret)}")

,RMSE ($),MAE ($),MAPE (%),Directional accuracy,Out-of-sample R2 (returns)
Naive (no change),0.679310,0.475575,0.858826,NaN,0.000000
"ARIMA(2, 0, 0)",0.680998,0.476389,0.860794,0.505896,-0.012681


ARIMA(2, 0, 0) Directional Hit Rate: 0.5059 (Binomial p-value vs 0.50: 0.6356)
Price Errors  -> DM Stat: 0.7955, p-value: 0.4264 | No significant difference
Return Errors -> DM Stat: 1.0951, p-value: 0.2736 | No significant difference


### Conclusion
- `ARIMA(2, 0, 0)` achieved an out-of-sample RMSE of **$0.6810** and MAE of **$0.4764**, nearly identical to the Naive benchmark ($0.6793 RMSE, $0.4756 MAE).
- Out-of-sample $R^2$ is **-0.0127** and directional accuracy is **50.59%** ($p = 0.6356$ vs. a 50% coin flip).
- The Diebold-Mariano test confirms **no statistically significant difference** between `ARIMA(2, 0, 0)` and the Naive benchmark ($p = 0.4264$ on prices; $p = 0.2736$ on returns), consistent with near-random-walk daily equity returns.